# 2. Préparation des données et construction des variables

**Projet :** *Yield Curve Forecasting: Are Some Maturities More Predictable Than Others?*

Ce notebook constitue la deuxième étape de notre étude. Après avoir exploré les dix séries de taux américains dans `01_data_exploration.ipynb`, nous préparons les observations nécessaires à une prévision **à un mois** pour chaque maturité.

Notre démarche suit les notions étudiées en cours : traitement des valeurs manquantes, construction de variables explicatives, séparation des données dans le temps et mise à l'échelle. Nous comparons également deux ensembles de variables — un ensemble restreint et un ensemble complet — afin de vérifier ultérieurement si davantage d'informations améliore réellement les prévisions.

Nous ne répétons ni les distributions, ni les graphiques historiques, ni les corrélations entre les taux déjà étudiés lors de l'exploration.

## 2.1. Chargement des séries

Nous partons des dix fichiers CSV téléchargés depuis FRED, correspondant aux maturités comprises entre trois mois et trente ans. Nous convertissons les dates au format temporel et les taux en valeurs numériques, puis nous réunissons les séries dans un même tableau.

Cette étape permet de repartir directement des données sources et d'exécuter ce notebook indépendamment de celui consacré à l'exploration.

In [ ]:
from pathlib import Path
from datetime import date
import numpy as np
import pandas as pd

SERIES = {
    "3M": "DGS3MO", "6M": "DGS6MO", "1Y": "DGS1", "2Y": "DGS2",
    "3Y": "DGS3", "5Y": "DGS5", "7Y": "DGS7", "10Y": "DGS10",
    "20Y": "DGS20", "30Y": "DGS30",
}
MATURITES = list(SERIES)
RETARDS = (1, 3, 6, 12)

# Fonctionne depuis la racine du projet ou depuis le dossier notebooks/.
RACINE = Path.cwd()
if not (RACINE / "data").is_dir():
    RACINE = RACINE.parent
DOSSIER_DATA = RACINE / "data"
DOSSIER_SORTIE = DOSSIER_DATA / "processed"

if not DOSSIER_DATA.is_dir():
    raise FileNotFoundError(f"Dossier data introuvable : {DOSSIER_DATA}")

series = {}
for maturite, code_fred in SERIES.items():
    chemin = DOSSIER_DATA / f"{code_fred}.csv"
    if not chemin.exists():
        raise FileNotFoundError(f"Fichier manquant : {chemin}")
    brut = pd.read_csv(chemin)
    colonne_date, colonne_taux = brut.columns[:2]
    dates = pd.to_datetime(brut[colonne_date], errors="coerce")
    taux = pd.to_numeric(brut[colonne_taux], errors="coerce")
    s = pd.Series(taux.to_numpy(), index=dates, name=maturite)
    s = s.loc[s.index.notna()].sort_index()
    if s.index.has_duplicates:
        raise ValueError(f"Dates dupliquées dans {chemin.name}")
    series[maturite] = s

donnees_journalieres = pd.concat(series, axis=1).sort_index()
donnees_journalieres.columns = MATURITES
donnees_journalieres.index.name = "Date"
print("Séries chargées :", len(donnees_journalieres.columns))
print("Période :", donnees_journalieres.index.min().date(), "→", donnees_journalieres.index.max().date())

## 2.2. Traitement des valeurs manquantes

Lors de l'exploration, nous avons constaté que certaines dates ne comportaient aucune observation, notamment lorsqu'aucun taux n'était publié. Nous choisissons de conserver uniquement les dates pour lesquelles les dix maturités sont renseignées. Nous pouvons ainsi construire une courbe des taux cohérente pour chaque observation retenue.

Le cours présente également l'imputation par moyenne ou médiane. Nous ne retenons pas ces méthodes ici, car elles introduiraient des valeurs artificielles à des dates sans publication. Nous ne supprimons pas non plus les fortes variations de taux uniquement parce qu'elles sont inhabituelles : elles peuvent correspondre à de véritables mouvements de marché.

In [ ]:
donnees_completes = donnees_journalieres.dropna(how="any").copy()
assert not donnees_completes.empty
assert donnees_completes.notna().all().all()
print("Données prêtes pour la conversion mensuelle.")

## 2.3. Passage à une fréquence mensuelle

Nous retenons la dernière observation disponible de chaque mois, à condition que les dix taux soient renseignés. Ce choix correspond à notre horizon de prévision : nous souhaitons utiliser l'information disponible à la fin du mois pour prévoir les taux du mois suivant.

Nous excluons le mois en cours s'il n'est pas terminé. Nous vérifions également que la série mensuelle ne présente pas de mois manquants, afin que le décalage d'une ligne corresponde bien à un mois calendaire.

In [ ]:
debut_mois_courant = pd.Timestamp(date.today().replace(day=1))
historique = donnees_completes.loc[donnees_completes.index < debut_mois_courant]
if historique.empty:
    raise ValueError("Aucun mois terminé disponible.")

mensuel = historique.groupby(historique.index.to_period("M")).tail(1).copy()
mensuel.index = mensuel.index.to_period("M").to_timestamp("M")
mensuel.index.name = "Mois"

# Vérification : aucune période calendaire ne doit manquer.
mois_attendus = pd.date_range(mensuel.index.min(), mensuel.index.max(), freq="ME")
mois_manquants = mois_attendus.difference(mensuel.index)
if len(mois_manquants):
    raise ValueError(f"Mois entiers manquants : {mois_manquants.tolist()}")
if len(mensuel) < 36:
    raise ValueError("Historique mensuel trop court pour cette méthode.")
print("Nombre de mois complets :", len(mensuel))

## 2.4. Construction des variables explicatives (*feature engineering*)

Nous construisons les variables à partir des taux mensuels. Contrairement aux rendements d'actions, nous utilisons des **variations de taux en points de base** plutôt que des rendements relatifs : une baisse de 4 % à 3 % correspond à une variation de −100 points de base.

Nous retenons les taux actuels, leurs valeurs retardées de 1, 3, 6 et 12 mois, les variations sur un mois, les moyennes mobiles sur 3 et 6 mois, ainsi que l'écart-type glissant des variations sur 3 mois. Nous ajoutons des indicateurs de forme de la courbe : niveau à 10 ans, pentes 10Y–3M et 10Y–2Y, et une mesure simple de courbure autour du 5Y.

Ces indicateurs sont calculés uniquement à partir des données disponibles à la fin du mois considéré. Nous conservons les fortes variations observées pendant les crises : elles ne sont pas automatiquement des erreurs de mesure.

In [ ]:
facteurs = pd.DataFrame(index=mensuel.index)
facteurs["niveau_10Y"] = mensuel["10Y"]
facteurs["pente_10Y_3M"] = mensuel["10Y"] - mensuel["3M"]
facteurs["pente_10Y_2Y"] = mensuel["10Y"] - mensuel["2Y"]
facteurs["courbure_5Y_3M_10Y"] = 2 * mensuel["5Y"] - mensuel["3M"] - mensuel["10Y"]

variables = pd.concat([mensuel.add_prefix("taux_"), facteurs], axis=1)

for retard in RETARDS:
    variables = pd.concat(
        [variables, mensuel.shift(retard).add_prefix(f"retard_{retard}m_")], axis=1
    )

variations = mensuel.diff()
for maturite in MATURITES:
    variables[f"variation_1m_bp_{maturite}"] = variations[maturite] * 100
    variables[f"moyenne_3m_{maturite}"] = mensuel[maturite].rolling(3).mean()
    variables[f"moyenne_6m_{maturite}"] = mensuel[maturite].rolling(6).mean()
    variables[f"volatilite_3m_bp_{maturite}"] = variations[maturite].rolling(3).std() * 100

colonnes_X = variables.columns.tolist()
print("Variables explicatives créées :", len(colonnes_X))

## 2.5. Définition des variables à prévoir

Notre problème est un problème de régression : nous cherchons à prédire des valeurs numériques, à savoir les taux d'intérêt du mois suivant.

Pour chaque observation du mois `t`, nous associons les variables explicatives disponibles à cette date aux dix taux observés au mois `t+1`. Par exemple, les données de janvier servent à prévoir les taux de fin février.

Les premières observations sont écartées lorsqu'elles ne disposent pas d'assez d'historique pour calculer les retards. La dernière observation est également exclue de l'entraînement, car son taux du mois suivant n'est pas encore disponible.

In [ ]:
cibles = mensuel.shift(-1).add_prefix("cible_")
colonnes_y = cibles.columns.tolist()
mois_cible = pd.Series(
    mensuel.index + pd.offsets.MonthEnd(1), index=mensuel.index, name="MoisCible"
)

dataset = pd.concat([variables, cibles], axis=1).dropna()
mois_cible = mois_cible.loc[dataset.index]

assert len(colonnes_y) == 10
assert len(dataset) > 50
assert dataset.notna().all().all()
assert (mois_cible > dataset.index).all()
print("Observations utilisables pour la prévision :", len(dataset))
print("Variables X :", len(colonnes_X), "| Cibles y :", len(colonnes_y))

## 2.6. Deux ensembles de variables à comparer

Avec environ 250 observations mensuelles après préparation, utiliser toutes les variables créées peut favoriser le surapprentissage. Nous définissons donc **deux ensembles de variables** qui seront comparés dans `03_modeling.ipynb` :

- **Ensemble restreint** : les dix taux contemporains, leurs dix variations mensuelles et les deux pentes de la courbe. Il conserve les informations financières principales avec moins de variables.
- **Ensemble complet** : les taux actuels et toutes les variables construites à la section précédente (retards, variations, moyennes mobiles, volatilités et indicateurs de courbe).

Nous ne choisissons pas à l'avance lequel sera le meilleur. Nous testerons les deux possibilités sur les mêmes périodes de validation. Les dix taux à prévoir sont identiques dans les deux cas.

In [ ]:
# Sous-ensemble interprétable, défini avant toute modélisation.
VARIABLES_SIMPLES = (
    [f"taux_{m}" for m in MATURITES]
    + [f"variation_1m_bp_{m}" for m in MATURITES]
    + ["pente_10Y_3M", "pente_10Y_2Y"]
)
VARIABLES_COMPLETES = colonnes_X.copy()

assert set(VARIABLES_SIMPLES).issubset(VARIABLES_COMPLETES)
assert len(VARIABLES_SIMPLES) == len(set(VARIABLES_SIMPLES))
print(f"Ensemble restreint : {len(VARIABLES_SIMPLES)} variables")
print(f"Ensemble complet : {len(VARIABLES_COMPLETES)} variables")

## 2.7. Séparation des données dans le temps

Nous séparons notre échantillon en trois parties : environ 70 % pour l'entraînement, 15 % pour la validation et 15 % pour le test.

Contrairement à l'exemple générique de `train_test_split` présenté en cours, nous ne mélangeons pas les observations. Dans notre cas, il est nécessaire de respecter l'ordre chronologique : les modèles doivent être entraînés sur des données antérieures à celles qu'ils prédisent.

Nous contrôlons aussi les dates des cibles. Une prévision réalisée à la fin d'un mois porte sur le mois suivant ; les observations situées à la frontière de deux ensembles sont donc traitées avec précaution pour que les cibles de l'entraînement ne se retrouvent pas dans la période de validation, et de même pour la validation et le test.

In [ ]:
n = len(dataset)
fin_train = int(n * 0.70)
fin_validation = int(n * 0.85)

train = dataset.iloc[:fin_train].copy()
validation = dataset.iloc[fin_train:fin_validation].copy()
test = dataset.iloc[fin_validation:].copy()

# Exclure les cibles qui chevauchent les périodes suivantes.
train = train.loc[mois_cible.loc[train.index] < validation.index.min()]
validation = validation.loc[mois_cible.loc[validation.index] < test.index.min()]

jeux = {"train": train, "val": validation, "test": test}
for nom, jeu in jeux.items():
    assert not jeu.empty
    print(f"{nom}: {len(jeu)} mois | {jeu.index.min():%Y-%m} → {jeu.index.max():%Y-%m}")

assert mois_cible.loc[train.index].max() < validation.index.min()
assert mois_cible.loc[validation.index].max() < test.index.min()
assert train.index.max() < validation.index.min() < test.index.min()

X_train, y_train = train[colonnes_X], train[colonnes_y]
X_val, y_val = validation[colonnes_X], validation[colonnes_y]
X_test, y_test = test[colonnes_X], test[colonnes_y]

## 2.8. Vérification des redondances entre les variables

Le notebook d'exploration étudiait les corrélations **entre les maturités**. Nous vérifions ici un autre aspect : les corrélations entre les **nouvelles variables explicatives**, notamment entre retards, niveaux et moyennes mobiles.

Cette vérification est effectuée **uniquement sur les données d'entraînement**, afin de ne pas utiliser la période de validation ou de test pour orienter nos choix. Nous comptons les paires de variables ayant une corrélation absolue supérieure à 0,98, sans les supprimer automatiquement : certaines redondances peuvent être utiles et les modèles régularisés peuvent les gérer.

Cette étape est un diagnostic, et non une sélection de variables guidée par les performances futures.

In [ ]:
# Diagnostic effectué uniquement sur Train, sans inspecter Validation ni Test.
correlations_train = X_train.corr().abs()
triangle_superieur = np.triu(np.ones(correlations_train.shape, dtype=bool), k=1)
paires_fortes = (
    correlations_train.where(triangle_superieur)
    .stack()
    .loc[lambda valeurs: valeurs > 0.98]
    .sort_values(ascending=False)
)

print(f"Paires très corrélées (|r| > 0,98) : {len(paires_fortes)}")
if len(paires_fortes):
    print("Exemples de paires :")
    display(paires_fortes.head(10).rename("corrélation absolue").to_frame())

# Nous ne retirons pas automatiquement de colonnes : l'ensemble restreint
# et l'ensemble complet seront évalués dans le prochain notebook.

## 2.9. Standardisation et encodage

Le cours présente plusieurs méthodes de mise à l'échelle, notamment `StandardScaler` et `MinMaxScaler`. Nous retenons **StandardScaler**, qui permet de centrer et réduire les variables explicatives. Cette transformation pourra être utile pour un modèle comme Ridge, sensible à l'échelle des variables. Nous ne jugeons pas nécessaire d'appliquer également `MinMaxScaler` : les deux approches sont des alternatives.

Pour éviter toute fuite d'information, nous calculons la moyenne et l'écart-type uniquement sur l'ensemble d'entraînement. Nous appliquons ensuite cette même transformation aux ensembles de validation et de test. Nous conservons les taux à prévoir dans leur unité d'origine pour interpréter directement les erreurs.

Enfin, notre jeu de données ne contient pas de variable explicative catégorielle. Nous n'utilisons donc ni `OneHotEncoder` ni `LabelEncoder`.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Démonstration : aucune transformation n'est ajustée sur validation/test.
scaler = StandardScaler()
X_train_standard = scaler.fit_transform(X_train)
X_val_standard = scaler.transform(X_val)
X_test_standard = scaler.transform(X_test)

assert np.isfinite(X_train_standard).all()
assert np.isfinite(X_val_standard).all()
assert np.isfinite(X_test_standard).all()
assert len(variables.select_dtypes(exclude="number").columns) == 0

print("StandardScaler : appliqué correctement sur les variables X, sans fichier supplémentaire.")
print("MinMaxScaler : non retenu (méthode alternative).")
print("Encodage catégoriel : non nécessaire.")

## 2.10. Enregistrement des jeux de données

Nous choisissons de limiter le nombre de fichiers produits. Les données finales sont enregistrées dans trois fichiers :

- `train.csv`, pour estimer les modèles ;
- `val.csv`, pour comparer les méthodes et ajuster leurs paramètres ;
- `test.csv`, pour évaluer les résultats sur une période qui n'a pas servi au choix du modèle.

Chaque fichier contient les variables explicatives et les dix cibles, dans les unités d'origine. Nous ne sauvegardons pas de copies standardisées : cette transformation sera intégrée au modèle concerné lors de l'étape suivante.

Les fichiers déjà présents dans `data/processed/` et issus d'anciennes exécutions ne sont pas supprimés automatiquement.

In [ ]:
DOSSIER_SORTIE.mkdir(parents=True, exist_ok=True)

for nom, jeu in jeux.items():
    assert jeu.notna().all().all()
    assert set(jeu.columns) == set(colonnes_X + colonnes_y)
    jeu.to_csv(DOSSIER_SORTIE / f"{nom}.csv", index_label="Mois")

print("Fichiers créés :")
for nom in jeux:
    fichier = DOSSIER_SORTIE / f"{nom}.csv"
    assert fichier.exists()
    print(f"- {fichier.name} ({fichier.stat().st_size:,} octets)")

print("Le preprocessing est terminé. Prochaine étape : 03_modeling.ipynb.")

## 2.11. Bilan de la préparation

Nous disposons désormais d'observations mensuelles associant les informations connues au mois `t` aux taux observés au mois `t+1`. Les périodes d'entraînement, de validation et de test sont ordonnées dans le temps, sans mélange aléatoire, et les observations situées aux frontières ont été traitées pour éviter d'utiliser une cible future dans le jeu précédent.

Nous retenons la suppression des dates sans publication, sans imputation par moyenne ou médiane. Nous avons défini **22 variables simples** et un **ensemble complet** comportant davantage d'indicateurs. Le diagnostic de corrélation effectué sur Train permettra de discuter la redondance de ces variables sans supprimer automatiquement des informations.

Pour les modèles sensibles à l'échelle, nous utiliserons `StandardScaler`, ajusté sur Train uniquement, idéalement au sein d'une `Pipeline`. Nous ne retenons pas `MinMaxScaler`, ni les encodages catégoriels, ni les méthodes de traitement de texte, car ils ne sont pas nécessaires à nos données.

Les trois fichiers exportés (`train.csv`, `val.csv` et `test.csv`) conservent l'ensemble des variables en unités d'origine. Dans `03_modeling.ipynb`, nous comparerons les ensembles de variables restreint et complet à l'aide des mêmes modèles et d'une prévision naïve. L'évaluation finale sera réalisée sur une période de test laissée de côté jusqu'au choix du modèle.